In [1]:
DEBUG = True

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import sys
import json

import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Setup paths
curr_dir = os.path.abspath(os.getcwd())
PROJECT_DIR = os.path.abspath(os.path.join(curr_dir, '..', '..', '..'))
Unlearn_Simple_DIR = os.path.join(PROJECT_DIR, 'Unlearn-Simple')
MUSE_DIR = os.path.join(Unlearn_Simple_DIR, 'MUSE')
MUSE_SRC_DIR = os.path.join(MUSE_DIR, 'src')
PROJECT_SRC_DIR = os.path.join(PROJECT_DIR, 'src')

# Ensure MUSE_DIR is first in sys.path for correct local imports
sys.path.insert(0, MUSE_DIR)
sys.path.append(MUSE_SRC_DIR)
sys.path.append(PROJECT_DIR)
sys.path.append(PROJECT_SRC_DIR)

# clean GPU mem
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Import the evaluation function from eval.py
import eval
from eval import load_then_eval_models
import eval_with_ILL
import input_loss_landscape.utils as input_loss_landscape_utils
from input_loss_landscape.utils import *

/home/liranc6/miniconda3/envs/unlearn_simple/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_or_privleak = 'raw'
base_original_sentences_neighbors_files = os.path.join(PROJECT_DIR, 'data', 'OPTML-Group-SimNPO-MUSE-News-llama-2-7b', 'top_k_20_n_tokens_30_k_neighbors_15', f'{raw_or_privleak}')

def get_first_neighbors_datasets(base_original_sentences_neighbors_files):
    types_map = ['holdout', 'retain', 'forget']
    datasets = {}
    neighbor_to_orig_map = {}
    for t in types_map:
        neighbor_to_orig_map[t] = {}
        file = os.path.join(base_original_sentences_neighbors_files, f'{t}.json')
        data = eval_with_ILL.read_json(file)
        first_neighbors = []
        for entry in data:
            # Each entry should have 'neighbors' as a list of dicts with 'text'
            if entry.get('neighbors') and len(entry['neighbors']) > 0:
                neighbor = entry['neighbors'][0]['text']
                first_neighbors.append(neighbor)
                orig = entry.get('text', None)
                if orig is not None:
                    neighbor_to_orig_map[t][neighbor] = orig
        datasets[t] = first_neighbors
    return datasets, neighbor_to_orig_map

first_neighbors_datasets, neighbor_to_orig_map = get_first_neighbors_datasets(base_original_sentences_neighbors_files)

# Print summary and a sample for each type
for split, neighbors in first_neighbors_datasets.items():
    print(f"{split}: {len(neighbors)} first neighbors")
    print("Sample:", neighbors[:3])
print("Sample neighbor_to_orig_map:", list(neighbor_to_orig_map.items())[:3])

holdout: 472 first neighbors
Sample: ['Last updated on .From the section Football\n\nStoke have signed West Brom striker Saido Berahino for a fee of £12m on a five-and-a-half-year deal.\n\n', 'Last updated on .From the section Football\n\nBBC Sport\'s football expert Mark Lawrenson will be making a prediction for all 380 Premier League games this season against a variety of guests.\n\nLawro\'s opponent for this week\'s Premier League fixtures is actor James McAvoy, star of new film \'Split\'.\n\nMcAvoy is a Celtic fan and says he grew up supporting them for many reasons.\n\n"I think your choice of football club quite often is not your choice," he told BBC Sport. "It is thrust upon you by your family, wherever you grew up, or sometimes even your religion, so it is a kind of environmental thing that you just soak up.\n\n"That is why I am a Celtic fan but why I enjoy being a Celtic fan is different and I have much more power over that.\n\n"In London, I keep an eye on Arsenal but I am not 

In [5]:
if DEBUG:
    subset_len = 200
else:
    subset_len = 500
    
datasets = {
    'forget': {'name': 'forget', 'data': first_neighbors_datasets['forget'][:subset_len]},
    'retain': {'name': 'retain', 'data': first_neighbors_datasets['retain'][:subset_len]},
    'holdout': {'name': 'holdout', 'data': first_neighbors_datasets['holdout'][:subset_len]}
}

In [6]:
# Model configuration
model_name = "OPTML-Group/SimNPO-MUSE-News-llama-2-7b"
tokenizer_dir = "meta-llama/Llama-2-7b-hf"

# Clean GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

print(f"Model loaded on device: {next(model.parameters()).device}")
print(f"Model type: {type(model)}")

# ILL strategy configuration
perc_of_tokens_to_replace = 0.1
n_tokens = int(300 * perc_of_tokens_to_replace)
strategy = {'name': 'embeddings', 'peak_top_k': 20, 'n_tokens': n_tokens, 'max_neighbors': 15}

# Path to precomputed cosine similarities
cosine_similarities_file = os.path.join(PROJECT_DIR, 'models', 'distilgpt2-finetuned-wikitext2', 'embeddings', 'token_knn_mapping_70_cosine.pth')

print(f"ILL Strategy: {strategy}")
print(f"Using cosine similarities from: {cosine_similarities_file}")

Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.03s/it]


Model loaded on device: cuda:0
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
ILL Strategy: {'name': 'embeddings', 'peak_top_k': 20, 'n_tokens': 30, 'max_neighbors': 15}
Using cosine similarities from: /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/models/distilgpt2-finetuned-wikitext2/embeddings/token_knn_mapping_70_cosine.pth


In [7]:
# Compute ILL features
print("Computing Input Loss Landscape features...")

output_dirs = os.path.join(f'{raw_or_privleak}', 'first_neighbor_with_all_losses')

new_ILL_eval_kwargs = {
    'model_name': model_name,
    'model': model,
    'tokenizer': tokenizer,
    'datasets': datasets,
    'prompt_column': 'text',
    'create_new_neighbors_file': False,
    'showplts': False,
    'cosine_similarities_file': cosine_similarities_file,
    'plots_output_dir': None,
    'strategy': strategy,
    'output_dirs': {'neighbors': output_dirs}
}

with open(cosine_similarities_file, 'rb') as f:
        cosine_similarities = torch.load(f)
        
if output_dirs:
    neighbors_subset = output_dirs
    
strategy_name = f"top_k_{strategy['peak_top_k']}_n_tokens_{strategy['n_tokens']}_k_neighbors_{strategy['max_neighbors']}".replace('[', '').replace(']', '').replace(',', '_').replace(' ', '')
model_name_without_slash = model_name.replace('/', '-')
output_file_base = os.path.join(PROJECT_PATH, 'data', f'{model_name_without_slash}', strategy_name)
if neighbors_subset:
    output_file_base = os.path.join(output_file_base, neighbors_subset)

max_neighbors = 15

Computing Input Loss Landscape features...


In [8]:
output_file_base

'/home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses'

In [9]:
def get_ILL_features_for_dataset(
    dataset_name,
    datasets,
    model,
    tokenizer,
    cosine_similarities,
    strategy,
    max_neighbors,
    output_file_base
):
    """
    Generate neighbors and extract ILL features for a given dataset name.
    Returns: features_tensor, feature_names, generated_data
    """
    dataset_data = datasets[dataset_name]['data']
    output_file = os.path.join(output_file_base, f'{dataset_name}.json')
    generated_data = generate_text_and_neighbors_from_dataset(
        model,
        tokenizer,
        dataset_data,
        prompt_column='text',
        cosine_similarities=cosine_similarities,
        max_new_tokens=300,
        num_min_tokens=20,
        strategy=strategy,
        num_proc=1,
        output_file=output_file,
        create_new_file=False,
    )
    features_tensor, feature_names = extract_features(generated_data, max_neighbors)
    neighbors_losses = torch.tensor(generated_data['mean_neighbors_loss']['losses'][:])
    new_feature_expanded = torch.cat([features_tensor, neighbors_losses], dim=1)
    new_feature_names = [f"neighbor_{i}_loss" for i in range(len(generated_data[0]['mean_neighbors_loss']['losses']))]
    feature_names = feature_names + new_feature_names
    return new_feature_expanded, feature_names, generated_data

get_ILL_features_for_dataset_kwargs = { 
    'datasets': datasets,
    'model': model,
    'tokenizer': tokenizer,
    'cosine_similarities': cosine_similarities,
    'strategy': strategy,
    'max_neighbors': max_neighbors,
    'output_file_base': output_file_base
}

datasets_names = ['forget', 'retain', 'holdout']
features = {}
for d_name in datasets_names:
    features_tensor, feature_names, generated_data = get_ILL_features_for_dataset(
        dataset_name=d_name,
        **get_ILL_features_for_dataset_kwargs
    )
    
    features[d_name] = {
        'unnormalized_features_tensor': features_tensor,
        'features_names': feature_names
    }

Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/forget.json already exists. Skipping generation.
Loaded 200 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/forget.json.


Batches: 100%|██████████| 47/47 [00:02<00:00, 22.49it/s]


Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/retain.json already exists. Skipping generation.
Loaded 200 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/retain.json.


Batches: 100%|██████████| 47/47 [00:01<00:00, 27.17it/s]


Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/holdout.json already exists. Skipping generation.
Loaded 200 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/OPTML-Group-SimNPO-MUSE-News-llama-2-7b/top_k_20_n_tokens_30_k_neighbors_15/raw/first_neighbor_with_all_losses/holdout.json.


Batches: 100%|██████████| 47/47 [00:01<00:00, 29.36it/s]


In [10]:
features['forget']['unnormalized_features_tensor'].shape

torch.Size([200, 30])

In [11]:
forget_tensor = features['forget']['unnormalized_features_tensor']
retain_tensor = features['retain']['unnormalized_features_tensor']
holdout_tensor = features['holdout']['unnormalized_features_tensor']


min_examples_num = min(subset_len,
                       len(forget_tensor), len(retain_tensor), len(holdout_tensor))

forget_tensor = forget_tensor[:min_examples_num]
retain_tensor = retain_tensor[:min_examples_num]
holdout_tensor = holdout_tensor[:min_examples_num]

forget_tensor.shape

torch.Size([200, 30])

In [29]:
print(f"{features['forget']['features_names']=}")

features['forget']['features_names']=['original_loss', 'mean_neighbor_loss', 'max_neighbor_loss', 'min_neighbor_loss', 'loss_variance', 'loss_std', 'mean_loss_increment', 'max_loss_increment', 'min_loss_increment', 'mean_gradient', 'max_gradient', 'gradient_variance', 'loss_volatility', 'local_curvature', 'increment_variance', 'neighbor_0_loss', 'neighbor_1_loss', 'neighbor_2_loss', 'neighbor_3_loss', 'neighbor_4_loss', 'neighbor_5_loss', 'neighbor_6_loss', 'neighbor_7_loss', 'neighbor_8_loss', 'neighbor_9_loss', 'neighbor_10_loss', 'neighbor_11_loss', 'neighbor_12_loss', 'neighbor_13_loss', 'neighbor_14_loss']


In [12]:
neighbor_to_orig_map.keys()

dict_keys(['holdout', 'retain', 'forget'])

In [ ]:
from datasets import Dataset

# Example: create a dataset from the original sentences (all values from neighbor_to_orig_map)
all_orig_sentences = []
origs_GT_dataset = {}
for split in neighbor_to_orig_map:
    origs_GT_dataset[split] = Dataset.from_dict({"text": list(neighbor_to_orig_map[split].keys())})

origs_GT_dataset['holdout']

Dataset({
    features: ['text'],
    num_rows: 472
})

In [ ]:
# take a subset of len subset_len of origs_dataset

subset_len = min(subset_len,
                 *[len(origs_GT_dataset[split]) for split in origs_GT_dataset])

for split in origs_GT_dataset:
    origs_GT_dataset[split] = origs_GT_dataset[split].select(range(subset_len))

In [17]:
neighbor_to_orig_map.keys()

dict_keys(['holdout', 'retain', 'forget'])

In [ ]:
origs_GT_dataset['holdout']

Dataset({
    features: ['text'],
    num_rows: 200
})

In [ ]:
def get_loss_and_perplexity(text, max_new_tokens=300):
    # assert isinstance(text['text'], str), "Input text must be a str"
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(text['text'], return_tensors="pt", padding=True, truncation=True, max_length=max_new_tokens).to(model.device)
        # labels = inputs["input_ids"]
        outputs_loss = model(**inputs, labels=inputs['input_ids'])
        loss = outputs_loss.loss
        # perplexity = torch.exp(loss)
    return {'loss': loss.item(), 
            # 'perplexity': perplexity.item()
            }

origs_GT_dataset_with_loss = {}
for split in origs_GT_dataset:
    origs_GT_dataset_with_loss[split] = origs_GT_dataset[split].map(
                                                                get_loss_and_perplexity,
                                                                batched=False,
                                                                num_proc=1,
                                                                desc="Calculating loss"
                                                            )

Calculating loss: 100%|██████████| 200/200 [00:08<00:00, 22.61 examples/s]


# Experiment - Data Extraction

In these experiments we try to extract the original sentence based on ite neighbor ILL

lets define terms:
- origin = 1st_neighbor_of_origin - the sentence we fed to the model (the target)
- origin_ground_truth = orig_GT - the sentence that the 1st_neighbor_of_origin was created from (by perturbation)
- neighbors - neighbors generated from the 1st_neighbor_of_origin

We want a model F such that F(1st_neighbor_of_origin)=x such that x is as close as possible to origin_ground_truth based on cosine similarity, (ofc, the model doesnt know origin_ground_truth)

I want ILL-Guided Reconstruction Model,
those are the stages:

1. Add the orig loss to the 1st_neighbor_of_origin neighbors and create new ILL
2. identify the new ILL stracture, and in it how the local surface of the origin_GT looks like (feature extraction)
3. using those featrues, make a classification model that tries to identify which of the neighbors is the original_GT

In [24]:
print(f"{origs_dataset_with_loss['forget']=}")

origs_dataset_with_loss['forget']=Dataset({
    features: ['text', 'loss'],
    num_rows: 200
})


In [26]:
print(f"{forget_tensor.shape=}")

forget_tensor.shape=torch.Size([200, 30])


In [ ]:
norm_forget_tensor, norm_retain_tensor, norm_holdout_tensor = eval_with_ILL.normalize_features(forget_tensor, retain_tensor, holdout_tensor)

In [30]:
print(f"{features.keys()=}")

features.keys()=dict_keys(['forget', 'retain', 'holdout'])
